# Dataset visualization & sanity checks

Run this **before training** on any newly collected `LeRobotDataset`. It answers the questions that otherwise cost you a wasted training run: are the joint/action signals sane, are the cameras in sync, is the language field populated, and what do the normalization stats look like?

> Requires the GPU/data machine with `lerobot` installed. On a laptop with no dataset, the last cell shows the same checks on synthetic data so the notebook still runs.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
REPO_ID = 'local/so101_tabletop_v1'  # <-- set to your dataset
ROOT = None  # or a local dataset directory

In [ ]:
from vla.data.lerobot_adapter import load_lerobot_dataset, build_episode_index, collect_low_dim_arrays
ds = load_lerobot_dataset(REPO_ID, root=ROOT)
index = build_episode_index(ds)
print(f'{index.num_episodes} episodes, {len(ds)} frames')
states, actions = collect_low_dim_arrays(ds, 'observation.state', 'action')
print('state shape', states.shape, 'action shape', actions.shape)

## 1. Action distribution per joint
A healthy teleop dataset has smooth, roughly unimodal per-joint action histograms. A spike at a joint's limit means the operator was fighting the workspace; a bimodal gripper is expected (open vs closed).

In [ ]:
names = ['shoulder_pan','shoulder_lift','elbow_flex','wrist_flex','wrist_roll','gripper']
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.hist(actions[:, i], bins=50)
    ax.set_title(names[i])
plt.tight_layout(); plt.show()

## 2. Action jerk (frame-to-frame delta)
Large jerk spikes are the fingerprint of dropped video frames or timestamp desync between cameras and joints — the #1 cause of a mysteriously-failing training run. This is the check that graduated into the `lerobot-lint-dataset` contribution idea.

In [ ]:
jerk = np.abs(np.diff(actions, axis=0)).max(axis=1)
plt.figure(figsize=(12,3)); plt.plot(jerk); plt.axhline(np.percentile(jerk,99), color='r', ls='--', label='99th pct')
plt.title('Max per-joint action delta per frame'); plt.legend(); plt.show()
print('frames above 99th pct jerk:', int((jerk > np.percentile(jerk,99)).sum()))

## 3. Sample camera frames + instruction
Eyeball a few frames to confirm the object is visible, the framing matches what you'll have at eval time, and the `task` string is the instruction you think it is.

In [ ]:
for fi in [0, len(ds)//2, len(ds)-1]:
    frame = ds[fi]
    front = np.asarray(frame['observation.images.front'])
    if front.shape[0] in (1,3): front = np.transpose(front, (1,2,0))
    plt.figure(figsize=(4,3)); plt.imshow(front); plt.axis('off')
    plt.title(str(frame.get('task','')) [:60]); plt.show()

## 4. Normalization stats (train split only)
These are the exact stats the model bakes into its checkpoint. Confirm no dimension has a near-zero std (a locked joint) that would blow up after division.

In [ ]:
from vla.data.normalization import NormStats
s = NormStats.from_array(states); a = NormStats.from_array(actions)
for i,n in enumerate(names):
    print(f'{n:14s} state mu={s.mean[i]:8.3f} sd={s.std[i]:7.3f} | action mu={a.mean[i]:8.3f} sd={a.std[i]:7.3f}')